In [1]:
from pathlib import Path
import json
import folium
from folium import Element
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import mapping
from branca.colormap import LinearColormap
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..')
DATA = ROOT / 'data' / 'processed'
FIGS = DATA / 'figures'
FIGS.mkdir(exist_ok=True)

STUDY_ECO_CODES = {'17', '18', '20', '21', '25', '43', '80'}

ECO_NAMES = {
    '17': 'Middle Rockies',
    '18': 'Wyoming Basin',
    '20': 'Colorado Plateaus',
    '21': 'Southern Rockies',
    '25': 'High Plains',
    '43': 'Northwestern Great Plains',
    '80': 'Northern Basin and Range',
}

ECO_COLORS = {
    '17': '#8B6914',
    '18': '#C6956A',
    '20': '#A0522D',
    '21': '#556B2F',
    '25': '#8B8B00',
    '43': '#4A7B6F',
    '80': '#7B68A0',
}

# bucket col in CSV is already title-cased
BUCKET_COLORS = {
    'Energy Generation':                 '#d4a017',
    'Energy Storage':                    '#d4a017',
    'Energy Transmission':               '#d4a017',
    'Nuclear Fuel Cycle':                '#d4a017',
    'Hydrological Restoration':          '#2d6a4f',
    'Terrestrial Ecosystem Restoration': '#2d6a4f',
    'Terrestrial Ecosystem':             '#2d6a4f',   # CSV truncated value
    'Settlement And Social':             '#7b6fa0',
    'Settlement Social':                 '#7b6fa0',   # CSV truncated value
    'Transport':                         '#7b6fa0',
}
BUCKET_RADIUS = {
    'Energy Generation':                 14,
    'Energy Storage':                    14,
    'Energy Transmission':               14,
    'Nuclear Fuel Cycle':                14,
    'Hydrological Restoration':          12,
    'Terrestrial Ecosystem Restoration': 12,
    'Terrestrial Ecosystem':             12,   # CSV truncated value
    'Settlement And Social':             10,
    'Settlement Social':                 10,   # CSV truncated value
    'Transport':                         10,
}
ROLE_COLORS = {
    'generation':   '#e25757',
    'load':         '#4a90d9',
    'transmission': '#f5a623',
    'mixed':        '#888888',
}

# ── Load data ──────────────────────────────────────────────────────────────
eco_gdf    = gpd.read_file(DATA / 'mw_ecoregions.geojson')
counties   = gpd.read_file(DATA / 'mw_counties.geojson')
actions_df = pd.read_csv(DATA / 'terra_wyoming_action_sequence.csv')
traj_df    = pd.read_csv(DATA / 'terra_wyoming_ees_trajectory.csv')
baseline   = pd.read_csv(DATA / 'mw_ecoregion_ees_summary.csv')
buses_gdf  = gpd.read_file(DATA / 'synthetic_buses.geojson')

# ── Dissolve ecoregion polygons to one feature per code (08c pattern) ──────
eco_gdf['US_L3CODE'] = eco_gdf['US_L3CODE'].astype(str)
eco_dissolved = eco_gdf[eco_gdf['US_L3CODE'].isin(STUDY_ECO_CODES)].dissolve(
    by='US_L3CODE', aggfunc='first'
).reset_index()
eco_wgs84 = eco_dissolved.to_crs(epsg=4326)

# ── Simplify county geometry for rendering speed ────────────────────────────
counties_simple = counties.copy()
counties_simple['geometry'] = counties_simple.geometry.simplify(0.01)

print(f'Ecoregion features (dissolved): {len(eco_wgs84)}')
print(f'Action rows: {len(actions_df)}')
print(f'Trajectory rows: {len(traj_df)}')
print(f'Buses: {len(buses_gdf)}')

Ecoregion features (dissolved): 7
Action rows: 13
Trajectory rows: 49
Buses: 500


In [2]:
# ── Prep: baseline lookup with ranks ─────────────────────────────────────
baseline_c = baseline.copy()
baseline_c['ecoregion_code'] = baseline_c['ecoregion_code'].astype(str)
baseline_c['rank'] = baseline_c['composite_score'].rank(
    ascending=False, method='min'
).astype(int)
n_eco = len(baseline_c)
base_lk = baseline_c.set_index('ecoregion_code')

# ── Prep: scenario data (year 2050) ────────────────────────────────────────
traj_df['ecoregion'] = traj_df['ecoregion'].astype(str)
traj_df['year'] = traj_df['year'].astype(int)
scen_year = 2050
scen_data = traj_df[traj_df['year'] == scen_year].copy()
if scen_data.empty:
    scen_year = int(traj_df['year'].max())
    scen_data = traj_df[traj_df['year'] == scen_year].copy()
    print(f'NOTE: year 2050 not found — using year {scen_year}')
scen_data = scen_data.set_index('ecoregion')
scen_data['composite'] = (scen_data['E'] + scen_data['Ec'] + scen_data['S']) / 3

# ── Map init ───────────────────────────────────────────────────────────────
m = folium.Map(location=[42.5, -108.5], zoom_start=5, tiles='CartoDB positron')

# ── Shared sequential colormap ─ same scale for both layers ────────────────
ees_colormap = LinearColormap(
    colors=['#f7fcf5', '#c7e9c0', '#74c69d', '#2d6a4f', '#1b4332'],
    vmin=0,
    vmax=10,
    caption='Composite EES Score (0–10)',
)
print(f'Colormap domain: {ees_colormap.vmin} – {ees_colormap.vmax}')

# ── Layer 0: Ecoregion labels (DivIcon at centroid) ────────────────────
ECO_SHORT = {
    '17': 'Middle Rockies',
    '18': 'Wyoming Basin',
    '20': 'Colorado Plateaus',
    '21': 'Southern Rockies',
    '25': 'High Plains',
    '43': 'NW Great Plains',
    '80': 'N Basin & Range',
}
label_group = folium.FeatureGroup(name='Layer 0 — Labels', show=True)
for _, row in eco_wgs84.iterrows():
    code = str(row['US_L3CODE'])
    short_name = ECO_SHORT.get(code, code)
    centroid = row.geometry.centroid
    folium.Marker(
        location=[centroid.y, centroid.x],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:10px;color:#1b1b1b;font-family:sans-serif;'
                f'font-weight:bold;text-shadow:1px 1px 2px white,-1px -1px 2px white;'
                f'white-space:nowrap;pointer-events:none;">{short_name}</div>'
            ),
            icon_size=(120, 20),
            icon_anchor=(60, 10),
        ),
    ).add_to(label_group)
label_group.add_to(m)

# ── Layer 1: EES Baseline choropleth ───────────────────────────────────────
layer1 = folium.FeatureGroup(name='Layer 1 — EES Baseline', show=True)
for _, row in eco_wgs84.iterrows():
    code = str(row['US_L3CODE'])
    if code not in base_lk.index:
        continue
    name  = ECO_NAMES.get(code, f'Eco {code}')
    brow  = base_lk.loc[code]
    E, Ec, S = float(brow['E_score']), float(brow['Ec_score']), float(brow['S_score'])
    comp  = float(brow['composite_score'])
    rank  = int(brow['rank'])
    fcol  = ees_colormap(comp)
    bar   = '█' * round(comp) + '░' * (10 - round(comp))
    tt = (
        f'<b>{name} — Baseline</b><br>'
        + '━' * 24 + '<br>'
        + f'Composite EES: <b>{comp:.2f}</b> / 10&nbsp; {bar}<br>'
        + f'Environmental: {E:.2f}<br>'
        + f'Economic:&nbsp;&nbsp;&nbsp;&nbsp;{Ec:.2f}<br>'
        + f'Social:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;{S:.2f}'
    )
    folium.GeoJson(
        {'type': 'Feature', 'geometry': mapping(row.geometry), 'properties': {}},
        style_function=lambda x, c=fcol: {
            'fillColor': c,
            'color': '#2d6a4f',
            'weight': 1.5,
            'fillOpacity': 0.70,
        },
        tooltip=folium.Tooltip(tt, sticky=True),
    ).add_to(layer1)
layer1.add_to(m)

# ── Layer 2: 2050 Scenario choropleth ──────────────────────────────────────
layer2 = folium.FeatureGroup(name=f'Layer 2 — {scen_year} SMR Scenario', show=False)
for _, row in eco_wgs84.iterrows():
    code = str(row['US_L3CODE'])
    if code not in base_lk.index or code not in scen_data.index:
        continue
    name  = ECO_NAMES.get(code, f'Eco {code}')
    brow  = base_lk.loc[code]
    srow  = scen_data.loc[code]
    E_b, Ec_b, S_b   = float(brow['E_score']),  float(brow['Ec_score']),  float(brow['S_score'])
    comp_b = float(brow['composite_score'])
    E_s, Ec_s, S_s   = float(srow['E']),  float(srow['Ec']),  float(srow['S'])
    comp_s = float(srow['composite'])
    dE, dEc, dS = E_s - E_b, Ec_s - Ec_b, S_s - S_b
    dComp = comp_s - comp_b
    fcol  = ees_colormap(comp_s)
    bar   = '█' * round(comp_s) + '░' * (10 - round(comp_s))
    tt = (
        f'<b>{name} — {scen_year} Scenario</b><br>'
        + '━' * 30 + '<br>'
        + f'Composite EES: <b>{comp_s:.2f}</b> / 10&nbsp; {bar}&nbsp;(Δ {dComp:+.2f} from baseline)<br>'
        + f'Environmental: {E_s:.2f}&nbsp;(Δ {dE:+.2f})<br>'
        + f'Economic:&nbsp;&nbsp;&nbsp;&nbsp;{Ec_s:.2f}&nbsp;(Δ {dEc:+.2f})<br>'
        + f'Social:&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;{S_s:.2f}&nbsp;(Δ {dS:+.2f})'
    )
    folium.GeoJson(
        {'type': 'Feature', 'geometry': mapping(row.geometry), 'properties': {}},
        style_function=lambda x, c=fcol: {
            'fillColor': c,
            'color': '#2d6a4f',
            'weight': 2.0,
            'fillOpacity': 0.70,
        },
        tooltip=folium.Tooltip(tt, sticky=True),
    ).add_to(layer2)
layer2.add_to(m)

# ── Layer 3: Action markers ─────────────────────────────────────────────────
# Keys match actual location_label values in terra_wyoming_action_sequence.csv
LOCATION_COORDS = {
    'NW Great Plains (bus 285)':                 (-105.9147, 42.9694),
    'Wyoming Basin (bus 282)':                   (-108.8689, 41.7227),
    'Wyoming Basin — Kemmerer (bus 286)':   (-110.8904, 41.9401),
    'Middle Rockies (bus 291)':                  (-109.6970, 44.3173),
    'WACM–PACE–PSCO (bus 285)':         (-105.9147, 42.9694),
    'NW Great Plains (eco 43)':                  (-104.5,    44.2  ),
    'Wyoming Basin — Green River (eco 18)':  (-109.466,  41.528),
    'Gillette, Campbell Co. (eco 43)':            (-105.502,  44.291),
    'Rock Springs + Kemmerer (eco 18)':           (-109.9,    41.7  ),
    'Cheyenne, Laramie Co. (eco 43)':             (-104.820,  41.140),
    'Laramie, Albany Co. (eco 25)':               (-105.591,  41.312),
    'Casper, Natrona Co. (eco 18)':               (-106.313,  42.867),
}

layer3 = folium.FeatureGroup(name='Layer 3 — Deployed Actions', show=True)
skipped, placed = [], 0

for _, act in actions_df.iterrows():
    loc_label = str(act['location_label'])
    coords = LOCATION_COORDS.get(loc_label)
    if coords is None:
        print(f"WARNING: '{loc_label}' not in LOCATION_COORDS (skipped)")
        skipped.append(loc_label)
        continue

    lon, lat = coords
    bucket = str(act['bucket'])
    color  = BUCKET_COLORS.get(bucket, '#888888')
    radius = BUCKET_RADIUS.get(bucket, 10)

    cost = float(act['est_cost_usd'])
    cost_str = (f'${cost/1e9:.2f}B' if cost >= 1e9
                else f'${cost/1e6:.0f}M' if cost >= 1e6
                else f'${cost:,.0f}')

    mat_lines = ''
    for col, label, unit in [
        ('steel_t',    'Steel',    't'),
        ('concrete_t', 'Concrete', 't'),
        ('land_ac',    'Land',     'ac'),
        ('labor_py',   'Labor',    'person-yr'),
        ('uranium_t',  'Uranium',  't'),
    ]:
        val = act.get(col, 0)
        if pd.notna(val) and float(val) > 0:
            mat_lines += f'&nbsp;&nbsp;{label}: {float(val):,.0f}&nbsp;{unit}<br>'
    if not mat_lines:
        mat_lines = '&nbsp;&nbsp;—<br>'

    popup_html = (
        '<div style="font-family:sans-serif;font-size:12px;line-height:1.6;">'
        f'<b style="font-size:13px;">{act["action_name"]}</b><br>'
        f'<span style="color:#5a5a5a;">{bucket}</span>'
        '<hr style="margin:4px 0;border-color:#e0e0d8;">'
        f'<b>Location:</b> {loc_label}<br>'
        f'<b>Scale:</b> {int(act["magnitude"]):,}&nbsp;{act["unit_label"]}<br>'
        f'<b>Operational:</b> {int(act["operational_year"])}<br>'
        f'<b>Est. Cost:</b> {cost_str}<br>'
        '<hr style="margin:4px 0;border-color:#e0e0d8;">'
        '<b>Materials:</b><br>'
        f'{mat_lines}'
        '<hr style="margin:4px 0;border-color:#e0e0d8;">'
        '<b>EES Capital Effects:</b><br>'
        f'&nbsp; ΔE&nbsp;=&nbsp;{float(act["delta_E"]):+.3f}'
        f'&nbsp; ΔEc&nbsp;=&nbsp;{float(act["delta_Ec"]):+.3f}'
        f'&nbsp; ΔS&nbsp;=&nbsp;{float(act["delta_S"]):+.3f}<br>'
        f'<span style="color:#5a5a5a;font-size:11px;">Basis: {act["coeff_basis"]}</span>'
        '</div>'
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        weight=2,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{act['action_name']} — {loc_label}",
    ).add_to(layer3)
    placed += 1

layer3.add_to(m)
print(f'Layer 3: {placed} markers placed, {len(skipped)} skipped')

# ── Layer 4: County outlines ────────────────────────────────────────────────
layer4 = folium.FeatureGroup(name='Layer 4 — Counties', show=False)
folium.GeoJson(
    counties_simple.__geo_interface__,
    style_function=lambda _: {
        'fillColor': 'none',
        'color': '#888888',
        'weight': 0.8,
        'fillOpacity': 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME', 'STUSPS'], aliases=['County', 'State']
    ),
).add_to(layer4)
layer4.add_to(m)

# ── Layer 5: Grid buses (study-area BAs only) ───────────────────────────────
STUDY_BAS = {'WACM', 'PSCO', 'PACE', 'WAUW', 'NWMT'}
layer5 = folium.FeatureGroup(name='Layer 5 — Grid Buses', show=False)
study_buses = buses_gdf[buses_gdf['ba_code'].isin(STUDY_BAS)].copy()
for _, bus in study_buses.iterrows():
    lon_b, lat_b = bus.geometry.x, bus.geometry.y
    role  = str(bus.get('role', 'mixed'))
    color = ROLE_COLORS.get(role, '#888888')
    folium.CircleMarker(
        location=[lat_b, lon_b],
        radius=3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=0.5,
        tooltip=f"Bus {bus.get('bus_id','?')} | {role} | {bus.get('ba_code','')}",
    ).add_to(layer5)
layer5.add_to(m)
print(f'Layer 5: {len(study_buses)} study-area buses plotted')

# ── Legend (lower-left) ────────────────────────────────────────────────────
legend_html = (
    '<div style="position:fixed;bottom:80px;left:20px;background:white;'
    'border:1px solid #e0e0d8;border-radius:6px;padding:10px 14px;'
    'font-family:sans-serif;font-size:12px;line-height:1.8;z-index:1000;'
    'max-width:220px;box-shadow:0 2px 6px rgba(0,0,0,0.12);">'
    '<b>Action Types</b><br>'
    '<span style="color:#d4a017;">&#9679;</span> Energy Infrastructure<br>'
    '<span style="color:#2d6a4f;">&#9679;</span> Ecological Restoration<br>'
    '<span style="color:#7b6fa0;">&#9679;</span> Social &amp; Settlement<br>'
    '<hr style="margin:4px 0;border-color:#e0e0d8;">'
    '<span style="color:#5a5a5a;font-size:11px;">'
    'Toggle layers using the control ↗<br>'
    'Click markers for action details<br>'
    'Color scale = Composite EES (0–10)'
    '</span></div>'
)

# ── Title bar (top-center) ─────────────────────────────────────────────────
title_html = (
    '<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);'
    'background:white;border:1px solid #e0e0d8;border-radius:6px;padding:8px 20px;'
    'font-family:sans-serif;font-size:14px;font-weight:bold;z-index:1000;'
    'box-shadow:0 2px 6px rgba(0,0,0,0.12);color:#1b1b1b;white-space:nowrap;">'
    'Wyoming Basin Transition Scenario &nbsp;|&nbsp;'
    '<span style="font-weight:normal;color:#5a5a5a;">'
    'Baseline → 2050 SMR Scenario &nbsp;·&nbsp; 13 Actions &nbsp;·&nbsp; $10.3B CAPEX'
    '</span></div>'
)

m.get_root().html.add_child(Element(legend_html))
m.get_root().html.add_child(Element(title_html))

# ── LayerControl ───────────────────────────────────────────────────────────
# ── Color scale bar + verification ───────────────────────────
ees_colormap.add_to(m)

# ── Verification: color assignments per ecoregion ────────────────────
print()
print(f'Colormap domain confirmed: vmin={ees_colormap.vmin}, vmax={ees_colormap.vmax}')
print()
hdr = f"{'Ecoregion':<26} {'Baseline':>7}  {'Base color':<10}  {'Scenario':>8}  Scen color"
print(hdr)
print('─' * len(hdr))
for code in sorted(base_lk.index):
    name_v = ECO_NAMES.get(code, f'Eco {code}')
    comp_b = float(base_lk.loc[code, 'composite_score'])
    col_b  = ees_colormap(comp_b)
    if code in scen_data.index:
        comp_s = float(scen_data.loc[code, 'composite'])
        col_s  = ees_colormap(comp_s)
    else:
        comp_s, col_s = float('nan'), 'n/a'
    print(f'{name_v:<26} {comp_b:>7.2f}  {col_b:<10}  {comp_s:>8.2f}  {col_s}')
print()

# ── LayerControl ─────────────────────────────────────────────
folium.LayerControl(collapsed=False, position='topright').add_to(m)
print('Map construction complete')

Colormap domain: 0 – 10
Layer 3: 13 markers placed, 0 skipped
Layer 5: 36 study-area buses plotted

Colormap domain confirmed: vmin=0, vmax=10

Ecoregion                  Baseline  Base color  Scenario  Scen color
─────────────────────────────────────────────────────────────────────
Middle Rockies                5.87  #5ba682ff       5.89  #5ba581ff
Wyoming Basin                 3.74  #9ed8afff       5.14  #70c199ff
Colorado Plateaus             3.89  #99d6adff       3.89  #99d6adff
Southern Rockies              6.40  #4c9371ff       6.40  #4c9371ff
High Plains                   5.13  #70c199ff       5.54  #65b28cff
Northwestern Great Plains     4.55  #83cda3ff       8.50  #255a43ff
Northern Basin and Range      4.34  #8ad0a6ff       4.34  #8ad0a6ff

Map construction complete


In [3]:
out_path = FIGS / 'terra_wyoming_scenario_map.html'
m.save(str(out_path))
size_mb = out_path.stat().st_size / 1_048_576

print('=' * 60)
print('OUTPUT: terra_wyoming_scenario_map.html')
print('=' * 60)
print(f'  Size:   {size_mb:.1f} MB')
print(f'  Layers: 5 (2 choropleth + actions + counties + buses)')
print(f'  Action markers: {len(actions_df)} (check vs. 13 expected)')
print(f'  Ecoregion polygons: {len(eco_wgs84)} baseline + {len(eco_wgs84)} scenario')
print()
print('Open in browser to verify:')
print(f'  {out_path}')
print()
print(f'✓ terra_wyoming_scenario_map.html saved — {size_mb:.1f} MB')
print(f'✓ {placed} action markers placed ({len(skipped)} skipped — location not in LOCATION_COORDS)')
print(f'✓ Baseline layer: {len(eco_wgs84)} ecoregion polygons')
print(f'✓ Scenario layer: {len(eco_wgs84)} ecoregion polygons (year={scen_year})')
print('✓ LayerControl: 6 layers registered (Labels + Baseline + Scenario + Actions + Counties + Buses)')
print(f'✓ Choropleth fix complete — shared sequential colormap, domain 0–10, {size_mb:.1f} MB')

OUTPUT: terra_wyoming_scenario_map.html
  Size:   10.5 MB
  Layers: 5 (2 choropleth + actions + counties + buses)
  Action markers: 13 (check vs. 13 expected)
  Ecoregion polygons: 7 baseline + 7 scenario

Open in browser to verify:
  ../data/processed/figures/terra_wyoming_scenario_map.html

✓ terra_wyoming_scenario_map.html saved — 10.5 MB
✓ 13 action markers placed (0 skipped — location not in LOCATION_COORDS)
✓ Baseline layer: 7 ecoregion polygons
✓ Scenario layer: 7 ecoregion polygons (year=2050)
✓ LayerControl: 6 layers registered (Labels + Baseline + Scenario + Actions + Counties + Buses)
✓ Choropleth fix complete — shared sequential colormap, domain 0–10, 10.5 MB
